In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
from data_curation import DataLoader, SafeGroupKFold
from i3l_ml import ML
from enums import *
import json
import pandas as pd
from sklearn.model_selection import GroupKFold

# Load data and separate in train, test and external set

In [43]:
dl = DataLoader()
ml = ML()

In [44]:
modes = [
    Mode.RWD, 
    Mode.DP, 
    Mode.FMRAD, 
    Mode.PYRAD, 
    Mode.GEN
]

dataset = dl.create_dataset(
    modes=modes, 
    outcome='OS_MONTHS',
    subanalysis=Subanalysis.CLASSIC
)

Loaded RWD data with shape: (2075, 12)
Loaded DP data with shape: (934, 769)
Loaded FMRAD data with shape: (899, 4097)
Loaded PYRAD data with shape: (881, 129)
Loaded GEN data with shape: (1705, 5)
Selecting FM-RAD features with LASSO...


In [45]:
with open('split.json', 'r') as f:
    split = json.load(f)

train_set = dataset[dataset['Subject'].isin(split['TRAIN_SET'])].set_index('Subject')
test_set = dataset[dataset['Subject'].isin(split['TEST_SET'])].set_index('Subject')
ext_set = dataset[dataset['Subject'].str.startswith('UOC')].set_index('Subject')

In [46]:
X_train, y_train = train_set.drop(columns=['OS_MONTHS']), train_set['OS_MONTHS']
X_test, y_test = test_set.drop(columns=['OS_MONTHS']), test_set['OS_MONTHS']
X_ext, y_ext = ext_set.drop(columns=['OS_MONTHS']), ext_set['OS_MONTHS']

In [47]:
outcome_name = Outcome.OS_6

y_train = dl.get_outcome(
    outcome=y_train,
    outcome_name=outcome_name,
)
y_test = dl.get_outcome(
    outcome=y_test,
    outcome_name=outcome_name,
)
y_ext = dl.get_outcome(
    outcome=y_ext,
    outcome_name=outcome_name,
)

In [48]:
with open('submodel_features.json', 'r') as f:
    submodel_features = json.load(f)
    submodel_features = [f for f in submodel_features if f in X_train.columns]

X_train = X_train.drop(columns=submodel_features)
X_test = X_test.drop(columns=submodel_features)
X_ext = X_ext.drop(columns=submodel_features)

In [49]:
X_train_imputed, imputer = dl.impute_df(X_train)
X_test_imputed, imputer = dl.impute_df(X_test, imputer=imputer)
X_ext_imputed, imputer = dl.impute_df(X_ext, imputer=imputer)

In [50]:
X_train_scaled, scaler, to_standard_normalize, to_log_normalize = dl.normalize(X_train_imputed)
X_test_scaled, _, _, _ = dl.normalize(X_test_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)
X_ext_scaled, _, _, _ = dl.normalize(X_ext_imputed, scaler=scaler, to_standard_normalize=to_standard_normalize, to_log_normalize=to_log_normalize)

231 features log normalized
987 features standardized
231 features log normalized
987 features standardized
231 features log normalized
987 features standardized


# Train and evaluate

In [51]:
train_folds = dl.get_loco_folds(pd.Series(train_set.index))
cv = SafeGroupKFold(n_splits=len(train_folds.unique()))
def get_split():
    return cv.split(X_train_scaled, y_train, groups=train_folds)

In [52]:
model = ml.train_model(
    X=X_train_scaled, 
    y=y_train,
    model_name=Model.RF,
    cv=get_split,
    select_features=True
)
selected_features = model.feature_names_in_

c:\Users\aferr\miniconda3\envs\ml\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [100, 150] was inferred to Integer(low=100, high=150, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(100, 150), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\aferr\miniconda3\envs\ml\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension [10, 15] was inferred to Integer(low=10, high=15, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(10, 15), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\aferr\Desktop\Albi\Code\i3lung\ml_lung\for_publication\data_curation.py:194: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will alway

In [53]:
from sklearn.metrics import roc_auc_score

In [54]:
X_train_scaled = X_train_scaled[selected_features]
X_test_scaled = X_test_scaled[selected_features]
X_ext_scaled = X_ext_scaled[selected_features]

In [57]:
train_auc = roc_auc_score(y_train, model.predict_proba(X_train_scaled)[:, 1])
test_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
ext_auc = roc_auc_score(y_ext, model.predict_proba(X_ext_scaled)[:, 1])

print(f"Train AUC: {train_auc:.3f}")
print(f"Test AUC: {test_auc:.3f}")
print(f"External AUC: {ext_auc:.3f}")

Train AUC: 0.895
Test AUC: 0.580
External AUC: 0.644
